# 09 — Model 5: Transformer (top rung of the complexity ladder)

Top rung of the Sleep-EDF complexity ladder and the **only non-convolutional model**. Models 2–4 were CNNs
(shallow ~8.2K, medium ~93K, deep ~864K params); Model 5 is the **Transformer encoder** from
`harness/models/transformer.py` (~1.2M params), trained on the **raw** single-channel EEG (no band features,
no bandpass filter — only the loader's per-epoch z-normalisation).

**Controlled ladder.** Only the architecture changes from Model 4. Data split, test set, stopping rule,
seeds, device, batch size, optimizer and the multi-seed helper are all held identical, so any difference in
results (including faithfulness later) is attributable to the model, not to a confounded training change. The
transformer trains on the **unmodified ladder-wide protocol** — no learning-rate change, no warmup, no
optimizer change (verified in the pre-checks; see DECISIONS_LOG) — so the top rung is not a special case.

**Ladder context — two framings that travel with this model.**

1. *Receptive field.* The CNN rungs have a **bounded** receptive field that sweeps through the AASM
   stage-defining event scale (~500–1500 ms): Model 2's 460 ms at the lower edge, Model 3's 1060 ms inside
   it, Model 4's 2260 ms above it. The transformer instead has a **global** receptive field via
   self-attention — every 60-sample token attends to all 50 tokens — so it sits *outside* that RF-vs-event
   framing rather than extending it.

2. *Parameter gap = architecture family, not another parameter step.* At **1,204,741** params against Model
   4's **863,557**, this rung is only **~1.4×** above the previous one — the tightest gap on the ladder
   (Models 2→3 and 3→4 are ~11× and ~9×). So a **Model 4 vs Model 5 difference is better read as
   convolution-vs-attention (architecture family) than as another step along the parameter axis.** Keep that
   in mind when reading the comparison table below.

**Why patch_size = 60.** Each token embeds a 60-sample patch, giving 50 tokens — **one token per 60-sample
region of the harness RegionGrid**. That one-to-one token↔region alignment is deliberate: it lets the later
attention-weight analysis map onto the CMI region grid with **no aggregation step**, which is what makes the
attention analysis possible at Model 5.

**This notebook is TRAINING only.** The XAI + CMI faithfulness analysis (including the attention-weight
read) is a separate notebook, after all rungs are trained.

Sections: **1** setup · **2** data · **3** model + parameter count · **4** multi-seed training (you run it) ·
**5** learning verification + ladder comparison (you run it after training).

> **You run the training.** Section 4's LAUNCH cell times seed 0, prints a total-time estimate, then trains
> all 5 seeds unattended (MPS). This is the **largest and slowest rung** (~14 s/epoch in the pre-checks) —
> **check seed 0's estimate before letting the run continue** (MAX_EPOCHS is in §1, one place, to lower).

## 1. Setup, imports & config

In [ ]:
import sys
from pathlib import Path
# Locate repo root robustly: walk up to the dir containing sleep_edf/ (depth-independent).
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, precision_recall_fscore_support)

from sleep_edf.loader import load_sleep_edf                 # full test set (never subsampled)
from sleep_edf.validation import train_val_split            # fixed subject-level 17,742 / 2,258 split
import sleep_edf.config as cfg
from sleep_edf.training import run_all_seeds                # shared timed/progress/save driver
# Shared architecture + training recipe. train_cnn is architecture-agnostic (its name is historical);
# it trains the transformer on the identical recipe used by every CNN rung. CNN_VARIANTS is only needed
# to recompute the earlier CNN rungs' receptive fields in the §5 comparison.
from harness.models.transformer import build_transformer
from harness.models.cnn import (train_cnn, set_seed, torch_predict_proba,
                                count_parameters, CNN_VARIANTS)

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SEEDS       = [0, 1, 2, 3, 4]        # 5 seeds — fixed up front (see §4), no single-seed preview
MODEL_NAME  = "model5_transformer"
# Previous rungs (in ladder order), read from saved JSON for the comparison cell (§5).
PREV_MODEL_NAMES = ["model2_shallow_cnn", "model3_medium_cnn", "model4_deep_cnn"]

# ── Ladder-wide early-stopping protocol (fixed across Models 2-5; see DECISIONS_LOG) ──
# Monitor validation BALANCED ACCURACY, not loss: val loss is dominated by W+N2 (~69%),
# so a model can lower loss while N1/N3 recall degrades. These are passed to train_cnn
# (they are NOT harness defaults). MAX_EPOCHS is the one knob to revise after seed 0's timing.
# The transformer uses these UNCHANGED — no lr change, no warmup, no optimizer change.
STOP_MONITOR   = "val_balanced_accuracy"
STOP_MODE      = "max"
STOP_PATIENCE  = 10
STOP_MIN_DELTA = 0.002
MAX_EPOCHS     = 100          # <<< ceiling — change HERE (one place) after seeing seed 0's timing

# ── Training device (ladder-wide for Models 2-5) ──────────────────────────────
# MPS (Apple-Silicon GPU); ~8x faster than CPU for this transformer (pre-check: 7.2 vs 54 ms/step).
# Deterministic here despite dropout (bit-identical across repeats of a seed; verified in pre-checks).
# Falls back to CPU on non-Apple machines. Passed to train_cnn / torch_predict_proba; not a harness
# default. Numerics differ slightly from CPU (float32 across backends) — see DECISIONS_LOG.
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

OUT_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "metrics"
FIG_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "figures"
CKPT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "checkpoints"   # trained weights for the XAI notebook
for d in (OUT_DIR, FIG_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=3, suppress=True)
print("architecture: transformer | patch_size (from config):", cfg.TRANSFORMER_PATCH_SIZE, "| seeds:", SEEDS)
print(f"stopping rule: monitor={STOP_MONITOR} mode={STOP_MODE} patience={STOP_PATIENCE} "
      f"min_delta={STOP_MIN_DELTA} max_epochs={MAX_EPOCHS}")
print(f"training device: {DEVICE}"
      + ("  (Apple-Silicon GPU)" if DEVICE == "mps" else "  (MPS unavailable -> CPU fallback)"))
print("results ->", OUT_DIR, "| checkpoints ->", CKPT_DIR)

## 2. Data loading

`train_val_split()` (from `sleep_edf/validation.py`) returns the **fixed subject-level split** of the
identical 20,000-epoch subsample every ladder model uses: **17,742 train / 2,258 val**, with 6 whole
subjects held out for validation (leakage-free early stopping — see DECISIONS_LOG). It is *loaded, not
constructed here* — no resampling, no rebalancing. `load_sleep_edf("test")` is the **full** 40,145-epoch
test set, never subsampled. Raw signal only (per-epoch z-norm from the loader; no bandpass). Identical data
to Models 2–4.

In [ ]:
# Fixed subject-level split of the 20K subsample (train 17,742 / val 2,258) + full test set.
X_tr, y_tr, X_val, y_val = train_val_split()          # loads the frozen split; asserts its integrity
X_test, y_test           = load_sleep_edf("test")     # full 40,145, never subsampled

# Visible confirmation the right split loaded (per-class counts, train vs val).
print(f"train {X_tr.shape}  |  val {X_val.shape}  |  test {X_test.shape}")
print(f"\n{'stage':<6}{'train':>8}{'val':>8}{'test':>8}")
for c, cn in enumerate(CLASS_NAMES):
    print(f"{cn:<6}{int((y_tr==c).sum()):>8}{int((y_val==c).sum()):>8}{int((y_test==c).sum()):>8}")
print(f"{'TOTAL':<6}{len(y_tr):>8}{len(y_val):>8}{len(y_test):>8}")
# Guard: the fixed split must be exactly 17,742 / 2,258 (the committed ladder-wide numbers).
assert len(y_tr) == 17742 and len(y_val) == 2258, (len(y_tr), len(y_val))

# Reference points for the learning check (§5), from the TEST distribution.
test_counts = np.bincount(y_test, minlength=cfg.N_CLASSES)
floor = test_counts.max() / len(y_test); chance_balanced = 1.0 / cfg.N_CLASSES
print(f"\nmajority-class floor (always predict {CLASS_NAMES[int(test_counts.argmax())]}): {floor:.4f}"
      f"  | balanced-acc chance: {chance_balanced:.3f}")
print("N3 and N1 are the load-bearing minorities — keep them explicit in §5.")

## 3. Model definition & parameter count

The transformer is built from the **shared** `harness.models.transformer.build_transformer` — no architecture
code is copied into `sleep_edf/`. Its **patch_size comes from `sleep_edf/config.py`** (`TRANSFORMER_PATCH_SIZE
= 60` = one token per 60-sample RegionGrid region), **not** the harness default of 1 (which would tokenise
every timestep → 3000 tokens and O(3000²) attention). The cell below **hard-asserts the config patch_size
actually reached the patch embedding** (the `Conv1d` kernel *and* stride) — it raises rather than silently
training at patch 1, which would both blow up compute and break the token↔region alignment the attention
analysis depends on. Expected: **~1.2M params, 50 tokens (3000 / 60, no padding), positional embedding
50 × d_model** (not 3000 × d_model). Receptive field is **global** (self-attention), so unlike the CNNs there
is no finite RF to report.

In [ ]:
# Build the transformer with Sleep-EDF shapes; patch_size from config (NOT the harness default 1).
probe = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                          n_classes=cfg.N_CLASSES, patch_size=cfg.TRANSFORMER_PATCH_SIZE)

# HARD CHECK: the config patch_size reached the patch embedding (Conv1d kernel == stride == patch).
pk, ps_ = probe.embed.kernel_size[0], probe.embed.stride[0]
if not (pk == cfg.TRANSFORMER_PATCH_SIZE and ps_ == cfg.TRANSFORMER_PATCH_SIZE):
    raise RuntimeError(f"patch mismatch: config says {cfg.TRANSFORMER_PATCH_SIZE} but embed kernel/stride "
                       f"= {pk}/{ps_} — refusing to train at the wrong ladder point.")

n_params = count_parameters(probe)
n_tokens = probe.pos.shape[1]; d_model = probe.pos.shape[2]
pos_params = probe.pos.numel()

# No padding / truncation: 3000 divides evenly by the patch size.
assert cfg.INPUT_LENGTH % cfg.TRANSFORMER_PATCH_SIZE == 0, "input_length must divide by patch_size"
assert n_tokens == cfg.INPUT_LENGTH // cfg.TRANSFORMER_PATCH_SIZE

print(f"architecture=transformer | patch_size={cfg.TRANSFORMER_PATCH_SIZE} (= RegionGrid region size)")
print(f"tokens          : {n_tokens}  (= {cfg.INPUT_LENGTH} / {cfg.TRANSFORMER_PATCH_SIZE}, no padding/truncation)")
print(f"trainable params: {n_params:,}")
print(f"positional embed: {tuple(probe.pos.shape)} = {pos_params:,} params  (50 x d_model={d_model}, "
      f"NOT {cfg.INPUT_LENGTH} x d_model)")
print(f"receptive field : GLOBAL (self-attention; every token attends to all {n_tokens} tokens)")

# Flags against the recorded Model-5 ladder figures (~1.2M params, 50 tokens, pos-emb 50 x d_model).
if not (1190000 <= n_params <= 1220000):
    print(f"  !! FLAG: parameter count {n_params:,} is outside the expected ~1.2M — check before training.")
if n_tokens != 50:
    print(f"  !! FLAG: token count {n_tokens} != expected 50 — check patch_size before training.")
if probe.pos.shape[1] != 50:
    print(f"  !! FLAG: positional embedding has {probe.pos.shape[1]} rows, expected 50 (patch_size wrong?).")
del probe  # a fresh model is built per seed in §4

## 4. Multi-seed training (5 seeds — you run the LAUNCH cell)

**5 seeds, decided up front** (no single-seed preview): the transformer is trained from 5 random
initialisations to report mean ± spread. The shared driver `sleep_edf.training.run_all_seeds` times seed 0,
prints a total-time estimate, then trains the rest **unattended**, saving each seed to disk as it finishes
(`resume=True` skips seeds already saved).

**Training recipe.** Each seed builds a fresh transformer and trains it with the **shared** harness recipe
`train_cnn` — architecture-agnostic despite the name, the *identical* recipe used by every CNN rung (Adam
lr 1e-3, weight-decay 1e-4, batch 16, class-weighted cross-entropy). The per-seed function passes the
**ladder-wide stopping rule** set in §1 straight to `train_cnn` (`monitor='val_balanced_accuracy'`,
`mode='max'`, `patience=10`, `min_delta=0.002`, `max_epochs=100`) — **unchanged for the transformer**. Early
stopping tracks balanced accuracy on the fixed val split and restores the best-by-balanced-accuracy
checkpoint.

**Per seed we record** `stopped_epoch` and `stop_reason` (`patience` vs `max_epochs`), so §5 can show whether
any seed hit the ceiling (truncated while still improving). Each seed's trained weights are saved to
`results/checkpoints/` for the later XAI/attention notebook.

**Device.** Runs on `DEVICE` (§1: MPS; ~8× faster than CPU for this transformer). The pre-checks confirmed
MPS supports every op (no fallback) and is **deterministic per seed even with dropout** (bit-identical across
repeats of seed 0; seeds 0 vs 1 genuinely diverge), so the 5-seed spread is real seed variance. Numerics
differ slightly from CPU (float32 across backends), so results are not bit-comparable to a CPU run.

**Progress display note.** `train_cnn` exposes no per-epoch callback, so the driver's optional inner epoch
bar isn't driven — the **per-seed** outer bar and the seed-0 time estimate still show. (No harness changes.)

In [ ]:
def _predict_in_batches(pp, X, bs=512):
    """Batched inference: the harness predict_proba runs the whole array in one forward
    (fine for ECG200's ~100 samples); chunk it for Sleep-EDF's 40k test to avoid a huge
    intermediate activation. Pre-checks: a 512-row transformer chunk is ~1.35 GB on MPS, so
    40,145 -> 79 chunks fits comfortably. Not a training loop; does not touch the harness."""
    return np.concatenate([pp(X[i:i + bs]) for i in range(0, len(X), bs)], axis=0)

def train_one_seed(seed, tick=None):
    # `tick` is accepted for the driver contract but train_cnn has no per-epoch hook, so it
    # is never called (only the outer per-seed bar shows). Not modifying harness to add one.
    set_seed(seed)
    model = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                              n_classes=cfg.N_CLASSES, patch_size=cfg.TRANSFORMER_PATCH_SIZE)
    assert model.embed.kernel_size[0] == cfg.TRANSFORMER_PATCH_SIZE            # guard every seed
    info = train_cnn(model, X_tr, y_tr, X_val, y_val, seed=seed, device=DEVICE,   # shared recipe +
                     monitor=STOP_MONITOR, mode=STOP_MODE, patience=STOP_PATIENCE,  # ladder-wide
                     min_delta=STOP_MIN_DELTA, max_epochs=MAX_EPOCHS)               # stopping rule

    ckpt = CKPT_DIR / f"{MODEL_NAME}_seed{seed}.pt"                          # weights for the XAI notebook
    torch.save(model.state_dict(), ckpt)                                     # state_dict is device-agnostic

    pp = torch_predict_proba(model, device=DEVICE)   # model is on DEVICE -> predict there, returns CPU numpy
    yp = _predict_in_batches(pp, X_test).argmax(1)
    pr, rc, f1c, sup = precision_recall_fscore_support(
        y_test, yp, labels=list(range(cfg.N_CLASSES)), zero_division=0)
    return {
        "accuracy":          float(accuracy_score(y_test, yp)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, yp)),
        "macro_f1":          float(f1_score(y_test, yp, average="macro")),
        "n_params":          int(count_parameters(model)),
        "architecture":      "transformer",
        "patch_size":        int(cfg.TRANSFORMER_PATCH_SIZE),
        "n_tokens":          int(model.pos.shape[1]),
        "best_epoch":        int(info["best_epoch"]),        # best val balanced-accuracy epoch
        "stopped_epoch":     int(info["stopped_epoch"]),     # last epoch actually run
        "stop_reason":       info["stop_reason"],            # 'patience' | 'max_epochs'
        "best_val_bal_acc":  float(info["best_metric"]),     # monitored metric at the restored ckpt
        "checkpoint":        str(ckpt.relative_to(PROJECT_ROOT)),
        "confusion":         confusion_matrix(y_test, yp, labels=list(range(cfg.N_CLASSES))).tolist(),
        "per_class":         {CLASS_NAMES[c]: {"precision": float(pr[c]), "recall": float(rc[c]),
                                               "f1": float(f1c[c]), "support": int(sup[c])}
                              for c in range(cfg.N_CLASSES)},
    }

print("train_one_seed defined. Run the LAUNCH cell below to train all 5 seeds (unattended).")

### 4 ▶ LAUNCH TRAINING — run this one cell (unattended, MPS)

**This is the cell you run to train.** It times seed 0, prints an estimated total time for all 5 seeds
(clearly visible **before** the long run continues), then trains the rest automatically, saving each seed to
`sleep_edf/results/metrics/model5_transformer_seed{seed}.json` and its weights to
`sleep_edf/results/checkpoints/`. `resume=True` skips seeds already on disk — delete those files (or set
`resume=False`) to re-train fresh.

This is the **slowest rung** (~14 s/epoch in the pre-checks). After seed 0, check the printed timing: if a
full `max_epochs=100` run would be too long, stop, lower `MAX_EPOCHS` in §1, and re-launch.

In [ ]:
# ▶▶▶ LAUNCH: trains all 5 seeds unattended (times seed 0, estimates, then continues automatically) ◀◀◀
aggregate = run_all_seeds(
    train_one_seed, SEEDS, OUT_DIR, MODEL_NAME,
    summary_keys=["balanced_accuracy", "accuracy", "macro_f1", "stopped_epoch"],
    resume=True,        # delete model5_transformer_seed*.json (or resume=False) to force a fresh re-train
)

## 5. Learning verification (run after training)

Reads the saved aggregate (works after the LAUNCH cell or a kernel restart) and reports results **across the
5 seeds** (mean ± spread), never a single seed:

- **Balanced accuracy** against *both* references — the ~34% majority-class floor (always-W) and the 0.20
  five-class chance line.
- **Overall accuracy.**
- **Per-class recall and F1**, with **N1 and N3 explicit** (the load-bearing minorities).
- **Confusion matrix** (counts + row-normalised), same format as Models 1–4.
- **stopped_epoch / stop_reason across seeds** — if seeds routinely stop by `max_epochs`, they were
  truncated while still improving (raise `MAX_EPOCHS` in §1).

In [ ]:
import json
agg = json.load(open(OUT_DIR / f"{MODEL_NAME}_aggregate.json"))
per_seed = agg["per_seed"]; s = agg["summary"]

n_params = per_seed[str(SEEDS[0])]["n_params"]
print(f"Model 5 — transformer | params {n_params:,} | patch {per_seed[str(SEEDS[0])]['patch_size']} "
      f"-> {per_seed[str(SEEDS[0])]['n_tokens']} tokens | {len(SEEDS)} seeds "
      f"| trained on 17,742 (fixed 20K minus 6 val subjects)\n")

print(f"accuracy      : {s['accuracy']['mean']:.4f} ± {s['accuracy']['std']:.4f}   "
      f"(majority floor {floor:.4f}; margin {s['accuracy']['mean'] - floor:+.4f})")
print(f"balanced acc  : {s['balanced_accuracy']['mean']:.4f} ± {s['balanced_accuracy']['std']:.4f}   "
      f"(chance {chance_balanced:.3f}; margin {s['balanced_accuracy']['mean'] - chance_balanced:+.4f})")
print(f"macro-F1      : {s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f}")

print(f"\n{'stage':<6}{'recall':>9}{'f1':>9}{'precision':>11}{'test_support':>14}   (minority?)")
print('-' * 60)
for cn in CLASS_NAMES:
    R = np.mean([per_seed[str(sd)]['per_class'][cn]['recall']    for sd in SEEDS])
    F = np.mean([per_seed[str(sd)]['per_class'][cn]['f1']        for sd in SEEDS])
    P = np.mean([per_seed[str(sd)]['per_class'][cn]['precision'] for sd in SEEDS])
    sup = per_seed[str(SEEDS[0])]['per_class'][cn]['support']
    tag = "  <-- minority" if cn in ("N3", "N1") else ""
    print(f"{cn:<6}{R:>9.3f}{F:>9.3f}{P:>11.3f}{sup:>14}{tag}")

# stopped_epoch / stop_reason across seeds — is early stopping halting, or hitting the ceiling?
print(f"\n{'seed':<6}{'stopped_epoch':>15}{'stop_reason':>14}{'best_epoch':>12}{'best_val_bal':>14}")
print('-' * 61)
for sd in SEEDS:
    r = per_seed[str(sd)]
    print(f"{sd:<6}{r['stopped_epoch']:>15}{r['stop_reason']:>14}{r['best_epoch']:>12}{r['best_val_bal_acc']:>14.4f}")
n_ceiling = sum(per_seed[str(sd)]['stop_reason'] == 'max_epochs' for sd in SEEDS)
print(f"\nseeds stopped by max_epochs (ceiling): {n_ceiling}/{len(SEEDS)}"
      + ("  -> truncated while improving; raise MAX_EPOCHS in §1." if n_ceiling else "  -> all halted by patience (good)."))

In [ ]:
# Confusion matrix, summed over seeds — counts + row-normalised (recall per true stage). Model-1 format.
cm = np.sum([np.array(per_seed[str(sd)]['confusion']) for sd in SEEDS], axis=0)
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, M, title, fmt in [(ax[0], cm, "Confusion (counts, summed over seeds)", "d"),
                         (ax[1], cmn, "Row-normalised (recall per true stage)", ".2f")]:
    a.imshow(M, cmap="Blues", vmin=0, vmax=(M.max() if fmt == "d" else 1))
    for i in range(5):
        for j in range(5):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center", fontsize=8,
                   color="white" if M[i, j] > (M.max() * 0.5 if fmt == "d" else 0.5) else "black")
    a.set_xticks(range(5)); a.set_xticklabels(CLASS_NAMES)
    a.set_yticks(range(5)); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(title)
fig.suptitle("Model 5 (transformer) — confusion across 5 seeds", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "sleep_edf_09_model5_confusion.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_09_model5_confusion.png").relative_to(PROJECT_ROOT))

### Full ladder — Models 2, 3, 4, 5 (all four rungs)

All four rungs side by side, so the complexity ladder is visible end to end. Models 2–4 are read from their
saved aggregates — nothing hardcoded; each CNN receptive field is recomputed from the `variant`/`kernel_size`
recorded in its file. The transformer's receptive field is **global** (self-attention), shown as `global`
rather than a millisecond figure.

**Read the last step with care.** Params climb 8.2K → 93K → 864K → 1.20M, i.e. ~11× / ~9× / **~1.4×**. The
Model 4 → Model 5 step is the tightest on the ladder *and* the only change of architecture family, so a
difference there is best read as **convolution-vs-attention**, not as another move along the parameter axis.
Watch balanced accuracy, macro-F1 and the minority recalls (N1, N3) — this is the axis the faithfulness study
will read against.

In [ ]:
# Compare all four rungs. Earlier rungs' numbers come from their saved JSON (not hardcoded).
def _rf_from_variant(variant, k):
    rf, jump = 1, 1
    for _ in CNN_VARIANTS[variant]:
        rf += (k - 1) * jump; rf += (2 - 1) * jump; jump *= 2
    return rf

def _rung(agg_path):
    a = json.load(open(agg_path)); ps = a["per_seed"]; sm = a["summary"]; sds = [str(s) for s in a["seeds"]]
    p0 = ps[sds[0]]
    variant = p0.get("variant")   # CNN rungs have it; the transformer does not
    rf_ms = (_rf_from_variant(variant, p0["kernel_size"]) * 1000 // cfg.SAMPLING_RATE
             if variant in CNN_VARIANTS else None)          # None -> global (transformer)
    return {
        "name": a["model"], "params": p0["n_params"], "rf_ms": rf_ms,
        "bal": sm["balanced_accuracy"]["mean"], "bal_sd": sm["balanced_accuracy"]["std"],
        "mf1": sm["macro_f1"]["mean"],
        "n1_rec": float(np.mean([ps[s]["per_class"]["N1"]["recall"] for s in sds])),
        "n3_rec": float(np.mean([ps[s]["per_class"]["N3"]["recall"] for s in sds])),
    }

# ladder order: previous rungs (from saved JSON) then this model
rungs = [_rung(OUT_DIR / f"{m}_aggregate.json") for m in PREV_MODEL_NAMES]
rungs.append(_rung(OUT_DIR / f"{MODEL_NAME}_aggregate.json"))

def _rf_str(rf_ms): return f"{rf_ms}ms" if rf_ms is not None else "global"

print(f"{'rung':<22}{'params':>10}{'RF':>9}{'bal acc':>13}{'macro-F1':>10}{'N1 rec':>9}{'N3 rec':>9}")
print('-' * 82)
for r in rungs:
    print(f"{r['name']:<22}{r['params']:>10,}{_rf_str(r['rf_ms']):>9}"
          f"{r['bal']:>9.4f}±{r['bal_sd']:.3f}{r['mf1']:>10.4f}{r['n1_rec']:>9.3f}{r['n3_rec']:>9.3f}")
print('-' * 82)
# per-step deltas (rung N vs rung N-1); RF delta only meaningful between two bounded (CNN) rungs
for a, b in zip(rungs[:-1], rungs[1:]):
    label = f"Δ ({b['name'].split('_')[0]} - {a['name'].split('_')[0]})"
    rf_d = f"{b['rf_ms']-a['rf_ms']:+d}ms" if (a['rf_ms'] is not None and b['rf_ms'] is not None) else "—"
    print(f"{label:<22}{b['params']-a['params']:>+10,}{rf_d:>9}"
          f"{b['bal']-a['bal']:>+10.4f}{b['mf1']-a['mf1']:>+10.4f}"
          f"{b['n1_rec']-a['n1_rec']:>+9.3f}{b['n3_rec']-a['n3_rec']:>+9.3f}")

---

## 6. Side experiment — patch_size = 15 (resolution test)

> **This section is ADDED, not a replacement.** Everything above (the patch-60 model, its results,
> figures and saved JSON/checkpoints) is the ladder's Model 5 and stays intact. This section trains a
> **separate** transformer variant with its own `MODEL_NAME` and its own artifacts; nothing above is
> overwritten. Run §1 (setup) and §2 (data) first — this section reuses them.

**Why.** Model 5 at patch 60 (50 tokens) reached balanced accuracy **0.6603 ± 0.0049** — the *worst* rung on
the ladder, below even the 8,181-param shallow CNN (0.7133). The losses concentrate in **N1**
(recall 0.573 → 0.350) and **REM** (0.739 → 0.630), and those two bleed heavily into each other, while W and
N2 hold up. That pattern is consistent with **resolution loss**: a 600 ms patch is embedded by a *single
linear projection*, so fine within-patch structure is compressed before attention ever sees it — and N1/REM
are precisely the stages that depend on that fine structure.

**What this tests.** Whether the underperformance is a consequence of the **patching choice** or is
**architectural**. patch_size = 15 gives **200 tokens, 4 per 60-sample region**, at the signal's own
~15-sample autocorrelation grain (the same coherence scale the CNN kernel = 15 was set to; see DECISIONS_LOG,
region-size / kernel entries). Only `patch_size` changes — the split, test set, stopping rule, seeds, device,
batch size and helper are all identical to patch 60.

**Promotion criterion (stated in advance, before any result).** patch 15 is promoted to *the* Model 5 ladder
rung **only if balanced accuracy recovers into the CNN range (~0.71+)**. A marginal gain would **not** justify
the cost: patch 60 gives exactly **one token per CMI region** (attention weights map onto the region grid with
no aggregation), whereas patch 15 gives **four tokens per region**, requiring a documented within-region
aggregation step for the attention-weight analysis.

**Either way, patch 60 stays in the thesis.** If patch 15 recovers accuracy, that is evidence the
underperformance was a **resolution artifact** rather than attention being unsuited to the task; if it does
not, that strengthens the architectural reading. The patch-60-vs-patch-15 comparison belongs in the
discussion regardless of outcome.

### 6.1 Model — transformer at patch_size = 15

Same `build_transformer`, only `patch_size` differs. Expected: **200 tokens (3000 / 15, no padding)**,
positional embedding **200 × d_model** (vs patch-60's 50 × d_model), so the parameter count grows slightly.
Hard-asserts the patch reached the embedding, same as the patch-60 cell.

In [ ]:
import json   # this section reuses §1's imports; json is first needed here (patch-60 imports it in §5)
# Patch-15 variant config. Distinct MODEL_NAME -> separate JSON/checkpoints/figure (patch-60 untouched).
PATCH15            = 15
PATCH15_MODEL_NAME = "model5_transformer_patch15"

probe15 = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                            n_classes=cfg.N_CLASSES, patch_size=PATCH15)
# HARD CHECK: the patch size reached the patch embedding (Conv1d kernel == stride == patch).
pk, ps_ = probe15.embed.kernel_size[0], probe15.embed.stride[0]
if not (pk == PATCH15 and ps_ == PATCH15):
    raise RuntimeError(f"patch mismatch: expected {PATCH15} but embed kernel/stride = {pk}/{ps_} "
                       "— refusing to train at the wrong patch size.")

p15_params = count_parameters(probe15)
p15_tokens = probe15.pos.shape[1]; d_model = probe15.pos.shape[2]
assert cfg.INPUT_LENGTH % PATCH15 == 0, "input_length must divide by patch_size"
assert p15_tokens == cfg.INPUT_LENGTH // PATCH15

# patch-60 params for the side-by-side flag (read from its saved JSON, not hardcoded).
p60 = json.load(open(OUT_DIR / "model5_transformer_aggregate.json"))["per_seed"]["0"]["n_params"]

print(f"patch_size={PATCH15} (= 4 tokens per 60-sample CMI region)")
print(f"tokens          : {p15_tokens}  (= {cfg.INPUT_LENGTH} / {PATCH15}, no padding/truncation)")
print(f"trainable params: {p15_params:,}   (patch-60 was {p60:,}; Δ {p15_params - p60:+,} "
      f"from the {p15_tokens} x {d_model} positional embedding vs 50 x {d_model})")
print(f"receptive field : GLOBAL (self-attention over {p15_tokens} tokens)")
if p15_tokens != 200:
    print(f"  !! FLAG: token count {p15_tokens} != expected 200 — check patch_size.")
if not (1210000 <= p15_params <= 1230000):
    print(f"  !! FLAG: parameter count {p15_params:,} outside expected ~1.22M — check before training.")
del probe15

### 6.2 Multi-seed training (5 seeds — you run the LAUNCH cell)

Identical ladder-wide protocol to patch 60 — same fixed 17,742/2,258 split, untouched 40,145 test set,
stopping rule (`val_balanced_accuracy`, max, patience 10, min_delta 0.002, max_epochs 100), 5 seeds, MPS,
batch size, and `run_all_seeds` helper. **Only `patch_size` changed.**

> **Timing.** 200 tokens is **4× the sequence length** of patch 60, so self-attention costs **~16×** (O(N²))
> — expect substantially slower epochs than patch-60's ~14 s. The LAUNCH cell times seed 0 and prints a total
> estimate; **check it before letting the unattended run continue**, and lower `MAX_EPOCHS` (§1) if needed.

In [ ]:
def _predict_in_batches(pp, X, bs=512):   # redefined here so this section is self-contained
    return np.concatenate([pp(X[i:i + bs]) for i in range(0, len(X), bs)], axis=0)

def train_one_seed_p15(seed, tick=None):
    set_seed(seed)
    model = build_transformer(input_length=cfg.INPUT_LENGTH, in_channels=cfg.IN_CHANNELS,
                              n_classes=cfg.N_CLASSES, patch_size=PATCH15)
    assert model.embed.kernel_size[0] == PATCH15                              # guard every seed
    info = train_cnn(model, X_tr, y_tr, X_val, y_val, seed=seed, device=DEVICE,
                     monitor=STOP_MONITOR, mode=STOP_MODE, patience=STOP_PATIENCE,
                     min_delta=STOP_MIN_DELTA, max_epochs=MAX_EPOCHS)          # unchanged protocol

    ckpt = CKPT_DIR / f"{PATCH15_MODEL_NAME}_seed{seed}.pt"                    # distinct from patch-60
    torch.save(model.state_dict(), ckpt)

    pp = torch_predict_proba(model, device=DEVICE)
    yp = _predict_in_batches(pp, X_test).argmax(1)
    pr, rc, f1c, sup = precision_recall_fscore_support(
        y_test, yp, labels=list(range(cfg.N_CLASSES)), zero_division=0)
    return {
        "accuracy":          float(accuracy_score(y_test, yp)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, yp)),
        "macro_f1":          float(f1_score(y_test, yp, average="macro")),
        "n_params":          int(count_parameters(model)),
        "architecture":      "transformer",
        "patch_size":        int(PATCH15),
        "n_tokens":          int(model.pos.shape[1]),
        "best_epoch":        int(info["best_epoch"]),
        "stopped_epoch":     int(info["stopped_epoch"]),
        "stop_reason":       info["stop_reason"],
        "best_val_bal_acc":  float(info["best_metric"]),
        "checkpoint":        str(ckpt.relative_to(PROJECT_ROOT)),
        "confusion":         confusion_matrix(y_test, yp, labels=list(range(cfg.N_CLASSES))).tolist(),
        "per_class":         {CLASS_NAMES[c]: {"precision": float(pr[c]), "recall": float(rc[c]),
                                               "f1": float(f1c[c]), "support": int(sup[c])}
                              for c in range(cfg.N_CLASSES)},
    }

print("train_one_seed_p15 defined. Run the LAUNCH cell below to train the patch-15 variant (5 seeds).")

### 6.2 ▶ LAUNCH TRAINING (patch 15) — run this one cell (unattended, MPS)

Trains all 5 seeds of the patch-15 variant, saving to
`sleep_edf/results/metrics/model5_transformer_patch15_seed{seed}.json` and checkpoints to
`sleep_edf/results/checkpoints/` — **separate files from patch 60**. `resume=True` skips seeds already on
disk. **Watch seed 0's printed estimate** before letting it continue (this variant is ~16× the attention
cost of patch 60).

In [ ]:
# ▶▶▶ LAUNCH (patch 15): times seed 0, prints estimate, then trains the rest automatically ◀◀◀
aggregate_p15 = run_all_seeds(
    train_one_seed_p15, SEEDS, OUT_DIR, PATCH15_MODEL_NAME,
    summary_keys=["balanced_accuracy", "accuracy", "macro_f1", "stopped_epoch"],
    resume=True,        # delete model5_transformer_patch15_seed*.json (or resume=False) to re-train fresh
)

### 6.3 Learning verification — patch 15 (run after training)

Same format as §5, for the patch-15 aggregate: balanced accuracy vs the 0.3656 floor and 0.200 chance,
overall accuracy, macro-F1, per-class recall/F1/precision (N1 and N3 flagged), the per-seed stop table, and
the confusion matrix — the last is where the **N1↔REM** resolution hypothesis is most directly visible.

In [ ]:
agg15 = json.load(open(OUT_DIR / f"{PATCH15_MODEL_NAME}_aggregate.json"))
ps15 = agg15["per_seed"]; s15 = agg15["summary"]
n_params15 = ps15[str(SEEDS[0])]["n_params"]
print(f"Model 5 (patch 15) — transformer | params {n_params15:,} | patch {ps15[str(SEEDS[0])]['patch_size']} "
      f"-> {ps15[str(SEEDS[0])]['n_tokens']} tokens | {len(SEEDS)} seeds\n")
print(f"accuracy      : {s15['accuracy']['mean']:.4f} ± {s15['accuracy']['std']:.4f}   "
      f"(majority floor {floor:.4f}; margin {s15['accuracy']['mean'] - floor:+.4f})")
print(f"balanced acc  : {s15['balanced_accuracy']['mean']:.4f} ± {s15['balanced_accuracy']['std']:.4f}   "
      f"(chance {chance_balanced:.3f}; margin {s15['balanced_accuracy']['mean'] - chance_balanced:+.4f})")
print(f"macro-F1      : {s15['macro_f1']['mean']:.4f} ± {s15['macro_f1']['std']:.4f}")
print(f"\n>>> promotion criterion: balanced acc >= ~0.71 (CNN range) to promote patch 15 to the ladder rung. "
      f"{'MET' if s15['balanced_accuracy']['mean'] >= 0.71 else 'NOT met'} at {s15['balanced_accuracy']['mean']:.4f}.")

print(f"\n{'stage':<6}{'recall':>9}{'f1':>9}{'precision':>11}{'test_support':>14}   (minority?)")
print('-' * 60)
for cn in CLASS_NAMES:
    R = np.mean([ps15[str(sd)]['per_class'][cn]['recall']    for sd in SEEDS])
    F = np.mean([ps15[str(sd)]['per_class'][cn]['f1']        for sd in SEEDS])
    P = np.mean([ps15[str(sd)]['per_class'][cn]['precision'] for sd in SEEDS])
    sup = ps15[str(SEEDS[0])]['per_class'][cn]['support']
    tag = "  <-- minority" if cn in ("N3", "N1") else ""
    print(f"{cn:<6}{R:>9.3f}{F:>9.3f}{P:>11.3f}{sup:>14}{tag}")

print(f"\n{'seed':<6}{'stopped_epoch':>15}{'stop_reason':>14}{'best_epoch':>12}{'best_val_bal':>14}")
print('-' * 61)
for sd in SEEDS:
    r = ps15[str(sd)]
    print(f"{sd:<6}{r['stopped_epoch']:>15}{r['stop_reason']:>14}{r['best_epoch']:>12}{r['best_val_bal_acc']:>14.4f}")
n_ceiling = sum(ps15[str(sd)]['stop_reason'] == 'max_epochs' for sd in SEEDS)
print(f"\nseeds stopped by max_epochs (ceiling): {n_ceiling}/{len(SEEDS)}"
      + ("  -> truncated while improving; raise MAX_EPOCHS in §1." if n_ceiling else "  -> all halted by patience."))

In [ ]:
# Confusion matrix (patch 15), summed over seeds — watch the N1<->REM cell vs patch 60.
cm = np.sum([np.array(ps15[str(sd)]['confusion']) for sd in SEEDS], axis=0)
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, M, title, fmt in [(ax[0], cm, "Confusion (counts, summed over seeds)", "d"),
                         (ax[1], cmn, "Row-normalised (recall per true stage)", ".2f")]:
    a.imshow(M, cmap="Blues", vmin=0, vmax=(M.max() if fmt == "d" else 1))
    for i in range(5):
        for j in range(5):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center", fontsize=8,
                   color="white" if M[i, j] > (M.max() * 0.5 if fmt == "d" else 0.5) else "black")
    a.set_xticks(range(5)); a.set_xticklabels(CLASS_NAMES)
    a.set_yticks(range(5)); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(title)
fig.suptitle("Model 5 (transformer, patch 15) — confusion across 5 seeds", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "sleep_edf_09_model5_patch15_confusion.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_09_model5_patch15_confusion.png").relative_to(PROJECT_ROOT))

### 6.4 patch 60 vs patch 15 — direct comparison

Both transformer variants side by side: params, tokens, balanced accuracy, macro-F1, and **per-class recall
for all five stages** with deltas. **N1 and REM are surfaced explicitly** — the resolution hypothesis predicts
those two should recover if patching (not attention) caused the underperformance. patch-60 numbers are read
from its saved JSON, not hardcoded.

In [38]:
def _tf_rung(agg_path):
    a = json.load(open(agg_path)); ps = a["per_seed"]; sm = a["summary"]; sds = [str(s) for s in a["seeds"]]
    p0 = ps[sds[0]]
    rec = {cn: float(np.mean([ps[s]["per_class"][cn]["recall"] for s in sds])) for cn in CLASS_NAMES}
    return {"name": a["model"], "params": p0["n_params"], "tokens": p0["n_tokens"],
            "bal": sm["balanced_accuracy"]["mean"], "bal_sd": sm["balanced_accuracy"]["std"],
            "mf1": sm["macro_f1"]["mean"], "rec": rec}

p60 = _tf_rung(OUT_DIR / "model5_transformer_aggregate.json")          # patch 60 (the ladder rung)
p15 = _tf_rung(OUT_DIR / f"{PATCH15_MODEL_NAME}_aggregate.json")       # patch 15 (this experiment)

print(f"{'variant':<16}{'params':>11}{'tokens':>8}{'bal acc':>13}{'macro-F1':>10}")
print('-' * 58)
for r in (p60, p15):
    print(f"{r['name']:<16}{r['params']:>11,}{r['tokens']:>8}{r['bal']:>9.4f}±{r['bal_sd']:.3f}{r['mf1']:>10.4f}")
print(f"{'Δ (15 - 60)':<16}{p15['params']-p60['params']:>+11,}{p15['tokens']-p60['tokens']:>+8}"
      f"{p15['bal']-p60['bal']:>+13.4f}{p15['mf1']-p60['mf1']:>+10.4f}")

print(f"\nper-class recall:   {'patch60':>9}{'patch15':>9}{'Δ':>9}")
for cn in CLASS_NAMES:
    d = p15['rec'][cn] - p60['rec'][cn]
    flag = "   <-- resolution hypothesis" if cn in ("N1", "REM") else ""
    print(f"  {cn:<4}            {p60['rec'][cn]:>9.3f}{p15['rec'][cn]:>9.3f}{d:>+9.3f}{flag}")

print(f"\npromotion criterion (balanced acc >= ~0.71): "
      f"{'MET' if p15['bal'] >= 0.71 else 'NOT met'} at {p15['bal']:.4f}  "
      f"(patch 60 = {p60['bal']:.4f}).")

variant              params  tokens      bal acc  macro-F1
----------------------------------------------------------
model5_transformer  1,204,741      50   0.6603±0.005    0.6296
model5_transformer_patch15  1,218,181     200   0.6395±0.010    0.5989
Δ (15 - 60)         +13,440    +150      -0.0208   -0.0307

per-class recall:     patch60  patch15        Δ
  W                   0.838    0.826   -0.013
  N1                  0.350    0.316   -0.034   <-- resolution hypothesis
  N2                  0.674    0.601   -0.073
  N3                  0.810    0.802   -0.008
  REM                 0.630    0.653   +0.022   <-- resolution hypothesis

promotion criterion (balanced acc >= ~0.71): NOT met at 0.6395  (patch 60 = 0.6603).


### 6.5 Full 5-rung ladder — both candidate Model-5 stories

The whole ladder printed **twice**: once with **patch 60** as Model 5, once with **patch 15** as Model 5, so
both candidate stories are visible side by side. Models 2–4 are read from their saved JSON (CNN receptive
fields recomputed from the recorded variant/kernel); the transformer's RF is `global` either way.

In [ ]:
def _rf_from_variant(variant, k):
    rf, jump = 1, 1
    for _ in CNN_VARIANTS[variant]:
        rf += (k - 1) * jump; rf += (2 - 1) * jump; jump *= 2
    return rf

def _rung(agg_path):
    a = json.load(open(agg_path)); ps = a["per_seed"]; sm = a["summary"]; sds = [str(s) for s in a["seeds"]]
    p0 = ps[sds[0]]; variant = p0.get("variant")
    rf_ms = (_rf_from_variant(variant, p0["kernel_size"]) * 1000 // cfg.SAMPLING_RATE
             if variant in CNN_VARIANTS else None)
    return {"name": a["model"], "params": p0["n_params"], "rf_ms": rf_ms,
            "bal": sm["balanced_accuracy"]["mean"], "bal_sd": sm["balanced_accuracy"]["std"],
            "mf1": sm["macro_f1"]["mean"],
            "n1": float(np.mean([ps[s]["per_class"]["N1"]["recall"] for s in sds])),
            "n3": float(np.mean([ps[s]["per_class"]["N3"]["recall"] for s in sds]))}

def _print_ladder(model5_agg, title):
    rungs = [_rung(OUT_DIR / f"{m}_aggregate.json") for m in PREV_MODEL_NAMES]
    rungs.append(_rung(OUT_DIR / model5_agg))
    rf = lambda r: f"{r}ms" if r is not None else "global"
    print(f"\n=== {title} ===")
    print(f"{'rung':<24}{'params':>11}{'RF':>9}{'bal acc':>13}{'macro-F1':>10}{'N1 rec':>9}{'N3 rec':>9}")
    print('-' * 85)
    for r in rungs:
        print(f"{r['name']:<24}{r['params']:>11,}{rf(r['rf_ms']):>9}"
              f"{r['bal']:>9.4f}±{r['bal_sd']:.3f}{r['mf1']:>10.4f}{r['n1']:>9.3f}{r['n3']:>9.3f}")

_print_ladder("model5_transformer_aggregate.json",          "Ladder with Model 5 = transformer @ patch 60 (current rung)")
_print_ladder(f"{PATCH15_MODEL_NAME}_aggregate.json",        "Ladder with Model 5 = transformer @ patch 15 (candidate)")

### 6.6 — Outcome: criterion not met, patch 60 retained

**Promotion criterion (from §6, fixed before the run):** promote patch 15 to *the* Model 5 ladder rung only if
balanced accuracy recovers into the CNN range (**~0.71+**).

**Result — not met.** patch 15 reached balanced accuracy **0.6395 ± 0.0098** (macro-F1 **0.5989**), *below*
patch 60's **0.6603 ± 0.0049** and far short of ~0.71. **patch 60 is retained as the Model 5 ladder rung.**

**Resolution hypothesis — not supported.** It predicted the two resolution-sensitive stages, **N1 and REM**,
would recover under finer (15-sample) patching. Instead (5-seed mean recall, patch 60 → patch 15):

| stage | patch 60 | patch 15 | change |
|---|---|---|---|
| W | 0.838 | 0.826 | −0.012 |
| **N1** † | 0.350 | 0.316 | **−0.034 (worse)** |
| N2 | 0.674 | 0.601 | **−0.073 (dropped)** |
| N3 | 0.810 | 0.802 | −0.008 |
| **REM** † | 0.630 | 0.653 | +0.023 (marginal) |

† stage the resolution hypothesis predicted would recover.

N1 — the largest deficit — got **worse**, N2 fell substantially, and only REM ticked up marginally; net
balanced accuracy dropped. Finer patching did not recover the underperformance, so the **resolution** reading
is not supported and the **architectural** reading (attention on this task, not the patch size) is
strengthened. This side experiment is the recorded evidence for keeping patch 60.